# 极客项目 ch12：Arduino 光敏电阻（LDR）实时绘图

依赖：pyserial/pyaudio/RPi.GPIO + 对应硬件，代码为完整忠实实现。对应《Python Playground》第 12 章（Arduino 简介）：LDR 电路 → Arduino 读 A0/A1 → 串口 → Python pySerial + matplotlib 实时滚动曲线；Python 代码搬运自 `code/ch12_ldr.py`，Arduino 草图搬运自 `code/ch12_arduino_ldr.ino`。

## 书 12.4.2 Arduino 程序（ldr.ino）

setup() 里 Serial.begin(9600)；loop() 读 A0/A1 两路 analogRead，按"值 值\n"格式发串口，delay(50) 即每 50 ms 一行——波特率与 Python 端 AnalogPlot 一致。

In [ ]:
// ===== Arduino sketch：读 A0/A1 两路 LDR 模拟值，每 50 ms 发一行串口（复制到 Arduino IDE 烧录） =====
// ch12_arduino_ldr.ino
//
// 原书 ldr.ino：读 A0/A1 两路 LDR 模拟值，以"值 值\n"格式发串口（每 50 ms 一行）
// 引脚：LDR 接 A0、A1；Serial.begin(9600) 与 Python 端一致

void setup()
{
  // initialize serial comms
  Serial.begin(9600);
}

void loop()
{
  // read A0
  int val1 = analogRead(0);
  // read A1
  int val2 = analogRead(1);
  // print to serial
  Serial.print(val1);
  Serial.print(" ");
  Serial.print(val2);
  Serial.print("\n");
  // wait
  delay(50);
}

## 书 12.5–12.6 Python 代码（AnalogPlot 与 main）

AnalogPlot 用两个 deque 做滚动缓冲（pop 最老、appendleft 最新），update() 读串口一行转两个浮点值并刷新曲线（坏数据 try/except 忽略）；main() 组装 FuncAnimation（20 ms 一帧）并 plt.show()。运行：`python ch12_ldr.py --port COM3 [--N 100]`。

In [ ]:
# ===== 书 12.3 节：所需模块 =====
import serial
import argparse
from collections import deque

import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
# ===== 书 12.5 节：AnalogPlot 类（构造 / add / addToDeq / update / close） =====
# plot class
class AnalogPlot:
    def __init__(self, strPort, maxLen):
        self.ser = serial.Serial(strPort, 9600)  # 波特率与 Arduino 端 Serial.begin 一致
        self.a0Vals = deque([0.0]*maxLen)
        self.a1Vals = deque([0.0]*maxLen)
        self.maxLen = maxLen

    # add data
    def add(self, data):
        assert len(data) == 2
        self.addToDeq(self.a0Vals, data[0])
        self.addToDeq(self.a1Vals, data[1])

    # add to deque, pop oldest value
    def addToDeq(self, buf, val):
        buf.pop()            # 从尾部删最老的
        buf.appendleft(val)  # 新值进头部 → 图形向右滚动

    # update plot
    def update(self, frameNum, a0, a1):
        try:
            line = self.ser.readline()
            data = [float(v) for v in line.decode().split()]  # Python 3：readline() 返回 bytes
            if len(data) == 2:
                self.add(data)
                a0.set_data(range(self.maxLen), self.a0Vals)
                a1.set_data(range(self.maxLen), self.a1Vals)
        except:
            pass  # 坏数据直接忽略
        return a0, a1

    # clean up
    def close(self):
        self.ser.flush()
        self.ser.close()

In [ ]:
# ===== 书 12.5/12.6 节：main()——argparse 与动画组装 =====
# main() function
def main():
    parser = argparse.ArgumentParser(description="LDR serial")
    parser.add_argument('--port', dest='port', required=True)
    parser.add_argument('--N', dest='maxLen', required=False)
    args = parser.parse_args()

    strPort = args.port
    print('reading from serial port %s...' % strPort)

    maxLen = 100
    if args.maxLen:
        maxLen = int(args.maxLen)

    analogPlot = AnalogPlot(strPort, maxLen)
    print('plotting data...')

    fig = plt.figure()
    ax = plt.axes(xlim=(0, maxLen), ylim=(0, 1023))
    a0, = ax.plot([], [])
    a1, = ax.plot([], [])
    anim = animation.FuncAnimation(fig, analogPlot.update,  # 存入 anim 变量：防回调被垃圾回收
                                   fargs=(a0, a1),
                                   interval=20)

    plt.show()
    analogPlot.close()
    print('exiting.')


if __name__ == '__main__':
    main()